In [1]:
import evaluate
from transformers import MT5Tokenizer, MT5ForConditionalGeneration
import pandas as pd

In [2]:
metric = evaluate.load("sacrebleu")

In [3]:
MODELNAME = "../v2/models/inference_lora_model"
PREFIX = "korrigiere: "
MAX_LENGTH = 128

tokenizer = MT5Tokenizer.from_pretrained(MODELNAME, legacy=False)
model = MT5ForConditionalGeneration.from_pretrained(MODELNAME)

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'T5Tokenizer'. 
The class this function is called from is 'MT5Tokenizer'.


In [4]:
df = pd.read_json("./data/chapter1-eval-v4.jsonl", lines=True)
data_corrupted = df["de_corrupted"].str.strip()
data_correct = df["de_correct"].str.strip()

data_corrupted = data_corrupted.tolist()#[:20]
data_correct = data_correct.tolist()#[:20]

In [5]:

inputs = [PREFIX + sentence for sentence in data_corrupted]

tokenized_inputs = tokenizer(
    inputs,
    max_length=MAX_LENGTH,
    truncation=True,
    padding="max_length",
    return_tensors="pt"
)


In [6]:
tokenized_inputs['input_ids'].shape

torch.Size([363, 128])

In [7]:

model.to('cuda')
tokenized_inputs = {k: v.to('cuda') for k, v in tokenized_inputs.items()}

# call in batches of 8
BATCH_SIZE = 8
all_outputs = []
for i in range(0, len(data_corrupted), BATCH_SIZE):
    batch_input_ids = tokenized_inputs['input_ids'][i:i+BATCH_SIZE]
    batch_attention_mask = tokenized_inputs['attention_mask'][i:i+BATCH_SIZE]
    
    batch_output = model.generate(
        input_ids=batch_input_ids,
        attention_mask=batch_attention_mask,
        max_length=MAX_LENGTH,
        num_beams=5, 
        early_stopping=True,
        repetition_penalty=2.5
    )
    
    all_outputs.extend(batch_output)

# corrected_text = tokenizer.decode(output[0], skip_special_tokens=True)
# corrected_text

# for token_id in output[0]:
#     token = tokenizer.decode([token_id])
#     print(f"{token_id.item()}: '{token}'")

In [8]:
decoded_preds = [tokenizer.decode(ids, skip_special_tokens=True) for ids in all_outputs]
for s1,s2 in zip(decoded_preds, data_correct):
    print(f"{s1}: '{s2}'")

Wir gehen morgen um zehn los.: 'Wir gehen morgen um zehn los.'
Nicht kann faule Menschen Ich ausstehen.: 'Ich kann faule Menschen nicht ausstehen.'
Wie kannst du so einen Mann lieben?: 'Wie kannst du so einen Mann lieben?'
So kann mich niemand erkennen.: 'So kann mich niemand erkennen.'
Manchmal fühle ich mich wie ein mutterloses Kind.: 'Manchmal fühle ich mich wie ein mutterloses Kind.'
Was genau muss er tun?: 'Was genau muss er tun?'
Ich will nicht, dass du mich falsch verstehst.: 'Ich will nicht dass du mich falsch verstehst.'
Im Allgemeinen machen sie im Ausland Urlaub.: 'Im Allgemeinen machen sie im Ausland Urlaub.'
Gehen ihr oft ins Theater?: 'Geht ihr oft ins Theater?'
Wir müssen ehren Vorfahren unsere.: 'Wir müssen unsere Vorfahren ehren.'
Lch glaube, sie gibt zu viel für Kleider aus.: 'lch glaube, sie gibt zu viel für Kleider aus.'
Ich frühstucke nicht, weil ich meistens zu spät aufstehe und ich keine Zeit mehr dafür habe.: 'Ich frühstücke nicht, weil ich meistens zu spät aufs

In [9]:
result = metric.compute(predictions=decoded_preds, references=data_correct)
result

{'score': 87.5943857697069,
 'counts': [3815, 3222, 2749, 2327],
 'totals': [3939, 3576, 3213, 2851],
 'precisions': [96.85199289159685,
  90.1006711409396,
  85.55866791160909,
  81.62048404068747],
 'bp': 0.9914055131923701,
 'sys_len': 3939,
 'ref_len': 3973}